In [1]:
import os
import pandas as pd

import findspark
findspark.init()

from pyspark import SparkContext  
from pyspark.sql import SparkSession, Row  
from pyspark.sql.types import StringType, StructField, StructType, FloatType
from pyspark.sql.functions import regexp_replace, explode, split, lower, desc
from pyspark.sql.functions import udf, when, concat, lit, avg, count, col, sum

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.stat import Correlation
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor, GBTRegressor

# os.environ['PYSPARK_PYTHON'] = '/home/hadoop/anaconda3/envs/BDA/bin/python'
# os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/hadoop/anaconda3/envs/BDA/bin/python'

# 数据读取

In [2]:
# 创建 SparkContext 和 SparkSession
sc = SparkContext('local', 'spark_project')
sc.setLogLevel('WARN')
spark = SparkSession.builder \
    .appName("BA Airline Analyse") \
    .getOrCreate()

# 数据文件的HDFS路径
data_path = 'hdfs://localhost:9000/user/airline/processedData.csv'

# 定义CSV文件的表头和数据类型
header = ['OverallRating', 'ReviewHeader', 'TypeOfTraveller', 'SeatType', 'DateFlown', 'SeatComfort',
          'CabinStaffService', 'GroundService', 'Aircraft', 'Food&Beverages', 'Departure', 'Destination']
float_type = ['OverallRating', 'SeatComfort', 'CabinStaffService', 'GroundService', 'Food&Beverages']

# 根据不同特征的数据类型构建Schema
fields = [] 
for column in header:  
    if column in float_type:
        field = StructField(column, FloatType(),True)
    else:  
        field = StructField(column, StringType(), True)
    fields.append(field)   
# 使用fields列表构建Schema
schema = StructType(fields)
# 使用预定义的模式从指定路径读取CSV文件
data = spark.read.csv(data_path, schema = schema, header = True)
# 创建临时视图
data.createOrReplaceTempView('data')

2024-05-25 22:14:47,203 WARN util.Utils: Your hostname, cube-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
2024-05-25 22:14:47,205 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2024-05-25 22:14:48,360 INFO spark.SparkContext: Running Spark version 3.2.0
2024-05-25 22:14:48,826 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2024-05-25 22:14:48,909 INFO resource.ResourceUtils: ==============================================================
2024-05-25 22:14:48,910 INFO resource.ResourceUtils: No custom resources configured for spark.driver.
2024-05-25 22:14:48,910 INFO resource.ResourceUtils: ==============================================================
2024-05-25 22:14:48,910 INFO spark.SparkContext: Submitted application: spark_project
2024-05-25 22:14:48,942 INFO resource.ResourceProfile: Default ResourceProfile created, e

In [3]:
data.printSchema()

root
 |-- OverallRating: float (nullable = true)
 |-- ReviewHeader: string (nullable = true)
 |-- TypeOfTraveller: string (nullable = true)
 |-- SeatType: string (nullable = true)
 |-- DateFlown: string (nullable = true)
 |-- SeatComfort: float (nullable = true)
 |-- CabinStaffService: float (nullable = true)
 |-- GroundService: float (nullable = true)
 |-- Aircraft: string (nullable = true)
 |-- Food&Beverages: float (nullable = true)
 |-- Departure: string (nullable = true)
 |-- Destination: string (nullable = true)



In [4]:
data.show(5)

+-------------+--------------------+---------------+---------------+---------+-----------+-----------------+-------------+--------------+--------------+----------+-----------+
|OverallRating|        ReviewHeader|TypeOfTraveller|       SeatType|DateFlown|SeatComfort|CabinStaffService|GroundService|      Aircraft|Food&Beverages| Departure|Destination|
+-------------+--------------------+---------------+---------------+---------+-----------+-----------------+-------------+--------------+--------------+----------+-----------+
|          2.4|"""do not upgrade...|       Business|  Economy Class|      Nov|        2.0|              3.0|          1.0|          A320|           1.0|  Brussels|     London|
|          7.1|"""Flight was smo...| Couple Leisure| Business Class|      Nov|        3.0|              3.0|          4.0|          A320|           4.0|  Heathrow|     Dublin|
|          0.6|"""Customer Servi...| Couple Leisure|  Economy Class|      Nov|        1.0|              1.0|          1.

In [5]:
# 计算每列中空值的数量
null_counts = data.select([sum(col(column).isNull().cast("int")).alias(column) for column in data.columns])
# 显示每列的空值数量
null_counts.show()

+-------------+------------+---------------+--------+---------+-----------+-----------------+-------------+--------+--------------+---------+-----------+
|OverallRating|ReviewHeader|TypeOfTraveller|SeatType|DateFlown|SeatComfort|CabinStaffService|GroundService|Aircraft|Food&Beverages|Departure|Destination|
+-------------+------------+---------------+--------+---------+-----------+-----------------+-------------+--------+--------------+---------+-----------+
|           12|          12|             23|      24|       24|         93|               33|           24|      24|            84|       24|         24|
+-------------+------------+---------------+--------+---------+-----------+-----------------+-------------+--------+--------------+---------+-----------+



In [6]:
# 删除 data 中包含任何空值的行
data = data.dropna()

In [7]:
# 检查是否还有空值
null_counts = data.select([sum(col(column).isNull().cast("int")).alias(column) for column in data.columns])
null_counts.show()

+-------------+------------+---------------+--------+---------+-----------+-----------------+-------------+--------+--------------+---------+-----------+
|OverallRating|ReviewHeader|TypeOfTraveller|SeatType|DateFlown|SeatComfort|CabinStaffService|GroundService|Aircraft|Food&Beverages|Departure|Destination|
+-------------+------------+---------------+--------+---------+-----------+-----------------+-------------+--------+--------------+---------+-----------+
|            0|           0|              0|       0|        0|          0|                0|            0|       0|             0|        0|          0|
+-------------+------------+---------------+--------+---------+-----------+-----------------+-------------+--------+--------------+---------+-----------+



# 乘客特征分析

## 分析不同类型乘客的占比

In [8]:
# 使用groupBy和count计算不同类型游客的数量
traveller_type_count = data.groupBy("TypeOfTraveller").count()
# 计算总的评论数量
total_reviews = data.count()
# 添加一个新列来计算占比
traveller_type_ratio = traveller_type_count.withColumn(
    "Ratio", col("count") * 100.0 / total_reviews)

# 显示数据分析结果
traveller_type_ratio.show()

+---------------+-----+------------------+
|TypeOfTraveller|count|             Ratio|
+---------------+-----+------------------+
| Couple Leisure|  862| 35.22680833673886|
| Family Leisure|  312| 12.75030649775235|
|   Solo Leisure|  750| 30.64977523498161|
|       Business|  523|21.373109930527175|
+---------------+-----+------------------+



In [9]:
traveller_type_ratio.toPandas().to_csv('./data/traveller_type_analysis.csv', index=False)

## 分析商务和休闲乘客数随时间的变化趋势

In [10]:
# 筛选出 TypeOfTraveller 和 DateFlown 这两列
traveller_reviews = data.select("TypeOfTraveller", "DateFlown")

# 将 Couple Leisure, Family Leisure, Solo Leisure 合并为 Leisure
filtered_reviews = traveller_reviews.withColumn(
    "TypeOfTraveller",
    when(col("TypeOfTraveller").like("%Leisure%"), "Leisure")
    .otherwise(col("TypeOfTraveller"))
)
# 按旅客类型和时间分组，统计出行次数
traveller_analysis = filtered_reviews.groupBy(
    "TypeOfTraveller", "DateFlown").count().orderBy("TypeOfTraveller", "DateFlown")

# 显示数据分析结果
traveller_analysis.show()

+---------------+---------+-----+
|TypeOfTraveller|DateFlown|count|
+---------------+---------+-----+
|       Business|      Apr|   38|
|       Business|      Aug|   48|
|       Business|      Dec|   30|
|       Business|      Feb|   39|
|       Business|      Jan|   40|
|       Business|      Jul|   52|
|       Business|      Jun|   44|
|       Business|      Mar|   43|
|       Business|      May|   40|
|       Business|      Nov|   54|
|       Business|      Oct|   47|
|       Business|      Sep|   48|
|        Leisure|      Apr|  149|
|        Leisure|      Aug|  177|
|        Leisure|      Dec|  177|
|        Leisure|      Feb|  113|
|        Leisure|      Jan|  149|
|        Leisure|      Jul|  146|
|        Leisure|      Jun|  163|
|        Leisure|      Mar|  148|
+---------------+---------+-----+
only showing top 20 rows



In [11]:
traveller_analysis.toPandas().to_csv('./data/traveller_analysis.csv', index=False)

## 分析不同舱位乘客的评分分布

In [12]:
# 统计不同舱位等级对应的评分分布
seat_type = data.select('SeatType', 'OverallRating')

# 显示数据分析结果
seat_type.show()

+---------------+-------------+
|       SeatType|OverallRating|
+---------------+-------------+
|  Economy Class|          2.4|
| Business Class|          7.1|
|  Economy Class|          0.6|
|  Economy Class|          1.8|
|Premium Economy|          8.2|
|  Economy Class|          7.4|
|  Economy Class|          2.5|
|  Economy Class|          5.8|
|  Economy Class|          7.3|
|  Economy Class|          6.5|
|  Economy Class|          1.9|
|  Economy Class|          7.9|
|  Economy Class|          4.3|
| Business Class|          1.1|
|  Economy Class|          0.3|
|  Economy Class|          0.7|
| Business Class|          5.7|
|  Economy Class|          2.6|
| Business Class|          4.2|
|  Economy Class|          3.6|
+---------------+-------------+
only showing top 20 rows



In [13]:
seat_type.toPandas().to_csv('./data/seat_type_analysis.csv', index=False)

# 情感分析

## 分析常见表扬词和常见批评词

In [14]:
# 定义停用词列表
stopwords = ["a", "an", "the", "and", "but", "if", "or", "because", "as", "of", "at", "by", "for", "with", "about", "against",
             "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "be", "i",
             "am", "you", "are", "was", "were", "is", "my", "have", "in", "out", "on", "off", "over", "under", "", "review"]

# 删除 ReviewHeader 中的所有标点符号
airline_reviews = data.withColumn("ReviewHeader", regexp_replace(
    col("ReviewHeader"), "[^a-zA-Z0-9\\s]", ""))
# 将 OverallRating 大于等于6的评论视为积极评论，反之则视为消极评论
# 按 OverallRating 是否小于6，拆分成两个表
negative_reviews = airline_reviews.filter(col("OverallRating") < 6)
positive_reviews = airline_reviews.filter(col("OverallRating") >= 6)

# 计算每个表的词频并过滤停用词
# 对 negative_reviews 进行词频计算并过滤停用词
negative_words = negative_reviews.select(
    explode(split(lower(col("ReviewHeader")), " ")).alias("word"))
negative_word_counts = negative_words.filter(~col("word").isin(
    stopwords)).groupBy("word").count().orderBy(desc("count"))
# 对 positive_reviews 进行词频计算并过滤停用词
positive_words = positive_reviews.select(
    explode(split(lower(col("ReviewHeader")), " ")).alias("word"))
positive_word_counts = positive_words.filter(~col("word").isin(
    stopwords)).groupBy("word").count().orderBy(desc("count"))

print('Words frequency in negative reviews:')
# 显示数据分析结果
negative_word_counts.show()
print('Words frequency in positive reviews:')
# 显示数据分析结果
positive_word_counts.show()

Words frequency in negative reviews:
+----------+-----+
|      word|count|
+----------+-----+
|        ba|  149|
|   airways|  140|
|   british|  139|
|   service|  130|
|  customer|  128|
|       not|  124|
|   airline|   98|
|experience|   72|
|      very|   72|
|        no|   69|
|     worst|   60|
|    flight|   52|
|     again|   49|
|     class|   47|
|      food|   46|
|     never|   39|
|  business|   38|
|     seats|   38|
|       fly|   37|
|    budget|   34|
+----------+-----+
only showing top 20 rows

Words frequency in positive reviews:
+------------+-----+
|        word|count|
+------------+-----+
|     service|  112|
|        crew|  105|
|      flight|  104|
|        very|  104|
|        good|   95|
|     airways|   87|
|     british|   86|
|    customer|   80|
|  experience|   60|
|    friendly|   59|
|       cabin|   51|
|   excellent|   40|
|          ba|   36|
|        food|   34|
|       staff|   34|
|       seats|   28|
|       great|   28|
|professional|   25|
|  

In [15]:
negative_word_counts.toPandas().to_csv('./data/negative_reviews.csv', index=False)
positive_word_counts.toPandas().to_csv('./data/positive_reviews.csv', index=False)

## 探究情感分析与总体评分的相关性

In [16]:
# 定义一个UDF函数，计算情感得分
def sentiment_score(text):
    scores = analyzer.polarity_scores(text)
    return float(scores['compound'])

In [17]:
# 初始化VADER情感分析工具
analyzer = SentimentIntensityAnalyzer()

# 定义一个用户自定义函数 (UDF) 用于计算情感得分
sentiment_udf = udf(sentiment_score, FloatType())
# 选择 OverallRating 和 ReviewHeader 两列
sentiment_reviews = data.select("OverallRating", "ReviewHeader")
# 使用UDF计算情感得分，直接在操作中进行情感分析
sentiment_reviews = sentiment_reviews.withColumn(
    "SentimentScore", sentiment_udf(col("ReviewHeader")))
# 选择 OverallRating 和 SentimentScore 两列
sentiment_scores = sentiment_reviews.select("OverallRating", "SentimentScore")
# 计算每个评分级别和对应情感得分的频次
sentiment_counts = sentiment_scores.groupBy(
    "OverallRating", "SentimentScore").count()

# 显示数据分析结果
sentiment_counts.show()

+-------------+--------------+-----+
|OverallRating|SentimentScore|count|
+-------------+--------------+-----+
|          4.9|       -0.5256|    1|
|          1.4|       -0.4588|    1|
|          3.7|       -0.1027|    2|
|          8.6|        0.4754|    1|
|          7.7|        0.7925|    1|
|          8.1|         0.659|    1|
|          2.7|        0.4576|    1|
|          0.4|        -0.296|    2|
|         10.0|        0.4019|    2|
|          4.4|       -0.2732|    1|
|          7.6|        0.5106|    1|
|          7.4|        0.6249|    1|
|          2.8|        0.1027|    1|
|          3.8|       -0.2584|    1|
|          6.5|        0.4404|    1|
|          7.8|        0.6369|    1|
|          6.8|       -0.3182|    1|
|          8.2|        0.3612|    1|
|          3.4|        0.4019|    1|
|          3.6|       -0.7717|    1|
+-------------+--------------+-----+
only showing top 20 rows



In [18]:
sentiment_counts.toPandas().to_csv('./data/sentiment_analysis.csv', index=False)

# 航线表现分析

## 统计Top10热门出发地&目的地的航线

In [32]:
# 统计 'Departure' 列中每个单元出现的频次
departure_counts = data.groupBy("Departure").agg(
    count("*").alias("Departure_count"))
# 统计 'Destination' 列中每个单元出现的频次
destination_counts = data.groupBy("Destination").agg(
    count("*").alias("Destination_count"))

departure_counts = departure_counts.orderBy(desc("Departure_count")).limit(10)
top_departure_list = [row['Departure'] for row in departure_counts.collect()]
destination_counts = destination_counts.orderBy(
    desc("Destination_count")).limit(10)
top_destination_list = [row['Destination']
                        for row in destination_counts.collect()]

# 选取 'Departure' 列的值在 top_departure_counts 列表中的那些行
route_reviews = data.filter(col("Departure").isin(
    top_departure_list) & col("Destination").isin(top_destination_list))

route_reviews = route_reviews.groupBy(
    "Departure", "Destination").count().orderBy(desc("count"))
route_reviews.toPandas().to_csv('./data/hot_city_analysis.csv', index=False)
route_reviews.show()

+----------+------------+-----+
| Departure| Destination|count|
+----------+------------+-----+
|  New York|      London|   20|
|  Heathrow|   Singapore|   19|
|Manchester|    Heathrow|   18|
|  Heathrow|       Miami|   18|
|     Miami|    Heathrow|   17|
| Vancouver|      London|   16|
| Cape Town|      London|   16|
|    London|    New York|   16|
|  Heathrow|       Dubai|   15|
| Singapore|    Heathrow|   15|
|   Glasgow|    Heathrow|   15|
| Vancouver|    Heathrow|   14|
|    London|Johannesburg|   14|
|    London|   Cape Town|   13|
|    London|   Singapore|   12|
|  Heathrow|   Hong Kong|   12|
| Singapore|      London|   12|
|     Miami|      London|   11|
|  Heathrow|   Cape Town|   11|
|    London|   Hong Kong|   10|
+----------+------------+-----+
only showing top 20 rows



# 飞机体验分析

## 分析全部乘客的总体评分分布

In [20]:
# 统计乘客整体的打分情况
rating_distribution = data.groupBy("OverallRating").count()

# 显示数据分析结果
rating_distribution.show()

+-------------+-----+
|OverallRating|count|
+-------------+-----+
|          9.1|   17|
|          9.4|   19|
|          6.9|   12|
|          9.0|   18|
|          5.0|   18|
|          9.9|   17|
|          7.6|   19|
|          3.8|   24|
|          7.8|   29|
|          2.2|   24|
|          5.5|   12|
|          2.5|   34|
|          5.7|   10|
|          7.7|   28|
|          8.5|   18|
|          9.5|   19|
|          8.1|   26|
|          8.3|   15|
|          3.4|   33|
|          4.6|   17|
+-------------+-----+
only showing top 20 rows



In [21]:
rating_distribution.toPandas().to_csv('./data/rating_analysis.csv', index=False)

## 分析各项评分指标中排名最高的10款机型

In [22]:
# 按飞机型号分组，计算每项分数的平均值
airline_average_scores = data.groupBy("Aircraft").agg(
    avg(col("OverallRating")).alias("AverageOverallRating"),
    avg(col("SeatComfort")).alias("AverageSeatComfort"),
    avg(col("CabinStaffService")).alias("AverageCabinStaffService"),
    avg(col("GroundService")).alias("AverageGroundService"),
    avg(col("Food&Beverages")).alias("AverageFood&Beverages")
)

# 按各项分数从高到低排序
overall_rating_sorted = airline_average_scores.select(
    "Aircraft", "AverageOverallRating").orderBy(col("AverageOverallRating").desc())
seat_comfort_sorted = airline_average_scores.select(
    "Aircraft", "AverageSeatComfort").orderBy(col("AverageSeatComfort").desc())
cabin_staff_service_sorted = airline_average_scores.select(
    "Aircraft", "AverageCabinStaffService").orderBy(col("AverageCabinStaffService").desc())
ground_service_sorted = airline_average_scores.select(
    "Aircraft", "AverageGroundService").orderBy(col("AverageGroundService").desc())
food_beverages_sorted = airline_average_scores.select(
    "Aircraft", "AverageFood&Beverages").orderBy(col("AverageFood&Beverages").desc())

# 显示数据分析结果
print("Overall Rating Sorted:")
overall_rating_sorted.show()
print("Seat Comfort Sorted:")
seat_comfort_sorted.show()
print("Cabin Staff Service Sorted:")
cabin_staff_service_sorted.show()
print("Ground Service Sorted:")
ground_service_sorted.show()
print("Food & Beverages Sorted:")
food_beverages_sorted.show()

Overall Rating Sorted:
+--------------------+--------------------+
|            Aircraft|AverageOverallRating|
+--------------------+--------------------+
|       Boeing 787-10|                10.0|
|           Saab 2000|                10.0|
|   Boeing 777-236 ER|                10.0|
|            A321-200|                10.0|
|            B737-400|   9.199999809265137|
|          B777-300ER|   9.199999809265137|
|           SAAB 2000|   9.199999809265137|
|   B747-400 in retro|   9.100000381469727|
|                B767|   8.600000381469727|
|                A219|   8.300000190734863|
|      Boeing 787-900|   7.383333404858907|
|           A350-1000|                 7.0|
|                A318|   6.924999982118607|
|      Boeing 787-800|   6.566666603088379|
|          boeing 787|                 6.5|
|      Boeing 737-800|   6.349999904632568|
|             Embraer|  6.2000000238418576|
|Boeing 787 Dreaml...|   6.199999809265137|
|          Boeing 744|   6.199999809265137|
|        

In [23]:
overall_rating_sorted.toPandas().to_csv('./data/overall_rating_analysis.csv', index=False)
seat_comfort_sorted.toPandas().to_csv('./data/seat_comfort_analysis.csv', index=False)
cabin_staff_service_sorted.toPandas().to_csv('./data/staff_service_analysis.csv', index=False)
ground_service_sorted.toPandas().to_csv('./data/ground_service_analysis.csv', index=False)
food_beverages_sorted.toPandas().to_csv('./data/food_beverages_analysis.csv', index=False)

## 分析热门机型中各项评分指标的分布

In [24]:
# 统计每个机型的数量，找出数量最多的5个机型
top_5_aircrafts = data.groupBy("Aircraft").agg(
    count("*").alias("count")).orderBy(col("count").desc()).limit(5)

# 筛选出这5个机型的数据
top_5_aircrafts_list = [row['Aircraft'] for row in top_5_aircrafts.collect()]
aircrafts_reviews = data.filter(col("Aircraft").isin(top_5_aircrafts_list))

# 按飞机型号分组，计算五项评分的平均值
aircrafts_average_scores = aircrafts_reviews.groupBy("Aircraft").agg(
    (avg(col("OverallRating"))/2).alias("AverageOverallRating"),
    avg(col("SeatComfort")).alias("AverageSeatComfort"),
    avg(col("CabinStaffService")).alias("AverageCabinStaffService"),
    avg(col("GroundService")).alias("AverageGroundService"),
    avg(col("Food&Beverages")).alias("AverageFood&Beverages")
)

# 显示数据分析结果
aircrafts_average_scores.orderBy(col("Aircraft")).show()

+--------------+--------------------+------------------+------------------------+--------------------+---------------------+
|      Aircraft|AverageOverallRating|AverageSeatComfort|AverageCabinStaffService|AverageGroundService|AverageFood&Beverages|
+--------------+--------------------+------------------+------------------------+--------------------+---------------------+
|          A320|   2.470652174892957|2.7880434782608696|      3.3260869565217392|   2.932065217391304|   2.5597826086956523|
|          A380|  2.6103365305954447|3.2451923076923075|      3.4471153846153846|              3.0625|   2.8461538461538463|
|Boeing 747-400|   2.315601505059049| 2.763157894736842|      3.1842105263157894|  2.9849624060150375|    2.654135338345865|
|    Boeing 777|   2.250977197049204|2.8403908794788273|      3.1335504885993486|  2.9120521172638436|     2.50814332247557|
|Boeing 777-200|  2.5358895759321065| 2.852760736196319|       3.312883435582822|   3.184049079754601|   2.7116564417177913|


In [25]:
aircrafts_average_scores.toPandas().to_csv('./data/aircrafts_scores_analysis.csv', index=False)

## 基于spark MLlib组件的分析

In [26]:
# 选择感兴趣的列进行分析
dataset = data.select('OverallRating', 'TypeOfTraveller', 'SeatType', 'DateFlown', 'SeatComfort',
                      'CabinStaffService', 'GroundService', 'Food&Beverages')

# 将 Solo Leisure, Family Leisure 和 Couple Leisure 统一合并为 Leisure 类型
dataset = dataset.withColumn("TypeOfTraveller", when(dataset["TypeOfTraveller"] == "Solo Leisure", "Leisure")
                             .when(dataset["TypeOfTraveller"] == "Family Leisure", "Leisure")
                             .when(dataset["TypeOfTraveller"] == "Couple Leisure", "Leisure")
                             .otherwise(dataset["TypeOfTraveller"]))

# 分别将 TypeOfTraveller, SeatType 和 DateFlown 列中的字符串类别转换为数值索引，再将数值索引转换为独热编码向量
indexer_traveller = StringIndexer(
    inputCol="TypeOfTraveller", outputCol="TypeOfTravellerIndex")
encoder_traveller = OneHotEncoder(
    inputCol="TypeOfTravellerIndex", outputCol="TypeOfTravellerVec", dropLast=False)
indexer_seat = StringIndexer(inputCol="SeatType", outputCol="SeatTypeIndex")
encoder_seat = OneHotEncoder(
    inputCol="SeatTypeIndex", outputCol="SeatTypeVec", dropLast=False)
indexer_month = StringIndexer(inputCol="DateFlown", outputCol="DateFlownIndex")
encoder_month = OneHotEncoder(
    inputCol="DateFlownIndex", outputCol="DateFlownVec", dropLast=False)

# 将所有特征列组合成一个特征向量
assembler = VectorAssembler(
    inputCols=['TypeOfTravellerVec', 'SeatTypeVec', 'DateFlownVec', 'SeatComfort', 'CabinStaffService',
               'GroundService', 'Food&Beverages'],
    outputCol='features'
)

# 构建处理和模型流水线
pipeline = Pipeline(stages=[
    indexer_traveller, encoder_traveller,
    indexer_seat, encoder_seat,
    indexer_month, encoder_month,
    assembler
])

# 拟合数据并转换数据集
prepared_data = pipeline.fit(dataset).transform(dataset)
# 将数据分为训练集和测试集
train_data, test_data = prepared_data.randomSplit([0.7, 0.3], seed=337)
print('Number of Training data:', str(train_data.count()))
print('Number of Testing data:', str(test_data.count()))

Number of Training data: 1705
Number of Testing data: 742


### 变量的相关性

In [27]:
# 计算特征列之间的斯皮尔曼相关矩阵
cor_matrix = Correlation.corr(prepared_data, 'features', 'spearman').first()[0]
# 将相关矩阵转换为数组
cor_count = pd.DataFrame(cor_matrix.toArray())

2024-05-25 22:16:29,796 WARN netlib.InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
2024-05-25 22:16:29,808 WARN netlib.InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
/usr/local/spark/python/pyspark/sql/context.py:125: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [28]:
# 定义列名，对应于 features 向量中的各个特征
columns = ['TravelForLeisure', 'TravelForBusiness', 'Economy Class', 'Business Class', 'Premium Economy', 
           'First Class', 'DepartInOct', 'DepartInSep', 'DepartInAug', 'DepartInNov', 'DepartInDec', 'DepartInJun', 
           'DepartInJul', 'DepartInMar', 'DepartInJan', 'DepartInMay', 'DepartInApr', 'DepartInFeb', 'SeatComfort', 
           'CabinStaffService', 'GroundService', 'Food&Beverages']
cor_count.columns = columns
cor_count.to_csv('./data/correlation_analysis.csv', index=False)

### 回归评估器的训练

In [29]:
def train_and_test(model_text, model, train_data, test_data, evaluator):
    # 训练模型
    trained_model = model.fit(train_data)
    # 使用训练好的模型对测试数据进行预测
    predictions = trained_model.transform(test_data)
    # 评估预测结果
    rmse = evaluator.evaluate(predictions)
    print(f"{model_text} RMSE: {rmse}")

    # 获取模型的特征重要性
    feature_importances = trained_model.featureImportances.toArray()

    return rmse, feature_importances

In [30]:
# 创建评估器对象
evaluator = RegressionEvaluator(
    labelCol='OverallRating', predictionCol='prediction', metricName="rmse")

# 决策树回归
# 实例化决策树回归器对象
dt = DecisionTreeRegressor(labelCol="OverallRating", featuresCol="features")
# 训练和测试决策树模型
dt_rmse, dt_feature_importances = train_and_test(
    "Decision Tree", dt, train_data, test_data, evaluator)

# 随机森林回归
# 实例化随机森林回归器对象
rf = RandomForestRegressor(labelCol="OverallRating", featuresCol="features")
# 训练和测试随机森林模型
rf_rmse, rf_feature_importances = train_and_test(
    "Random Forest", rf, train_data, test_data, evaluator)

# 梯度提升树回归
# 实例化 GBT 回归器对象
gbt = GBTRegressor(labelCol="OverallRating", featuresCol="features")
# 训练和测试 GBT 模型
gbt_rmse, gbt_feature_importances = train_and_test(
    "GBT", gbt, train_data, test_data, evaluator)

Decision Tree RMSE: 1.5338466122627674
Random Forest RMSE: 1.465686211430829


GBT RMSE: 1.537277400647288


In [31]:
# 将 RMSE 结果保存到字典中
rmse = {
    "Model": ["Decision Tree", "Random Forest", "GBT"],
    "RMSE": [dt_rmse, rf_rmse, gbt_rmse]
}
rmse_count = pd.DataFrame(rmse)
rmse_count.to_csv('./data/rmse_analysis.csv', index=False)

# 将决策树结果保存到字典中
dt_importances = {
    "Featuresc": columns,
    "Importances": dt_feature_importances
}
dt_count = pd.DataFrame(dt_importances)
dt_count.to_csv('./data/decision_tree_analysis.csv', index=False)

# 将随机森林结果保存到字典中
rf_importances = {
    "Features": columns,
    "Importances": rf_feature_importances
}
rf_count = pd.DataFrame(rf_importances)
rf_count.to_csv('./data/random_forest_analysis.csv', index=False)

# 将 GBT 结果保存到字典中
gbt_importances = {
    "Features": columns,
    "Importances": gbt_feature_importances
}
gbt_count = pd.DataFrame(gbt_importances)
gbt_count.to_csv('./data/gbt_analysis.csv', index=False)